# Train Squad's own model on a free Colab GPU

Fine-tunes Qwen3.5-0.8B with LoRA to route jobs to Squad's crew, scores it on the fixed test set, and gives you one ~0.5 GB `fin-q4_k_m.gguf` file to download.

**First:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all. It takes about 30 to 45 minutes.

No API key is needed: the examples come from the hand-written seeds in `training/seeds/`.

In [ ]:
# 1. The GPU and the code
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, getpass
REPO = "github.com/cosmospan/squadcode"
BRANCH = "claude/project-thread-8o4ttl"
os.chdir("/content")
if not os.path.isdir("squadcode"):
    r = os.system(f"git clone -q --depth 1 -b {BRANCH} https://{REPO} squadcode")
    if r != 0:  # a private repo needs a GitHub token with read access
        token = getpass.getpass("GitHub token (read access to the repo): ")
        os.system(f"git clone -q --depth 1 -b {BRANCH} https://x-access-token:{token}@{REPO} squadcode")
%cd /content/squadcode/training

In [ ]:
# 2. Packages. Colab's own torchao is too old for the newest PEFT.
!pip install -q -U transformers peft trl datasets accelerate "torchao>=0.16.0"

In [ ]:
# 3. Training examples from the seeds (no API key)
!python build_data.py --out work/data/synth.jsonl

In [ ]:
# 4. Train (about 15 to 30 minutes on a T4). The teacher's examples
# (data/teacher.jsonl, in the repo) join the seeds when they're there.
import os
DATA = "work/data/synth.jsonl" + (" data/teacher.jsonl" if os.path.exists("data/teacher.jsonl") else "")
print("Training on:", DATA)
!python train_lora.py --data {DATA} --out work/fin-lora

In [ ]:
# 5. Score it on the fixed test set, as the app asks it
import json, os, sys, torch
# Colab's Python may not be in the training folder after a restart.
os.chdir("/content/squadcode/training")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
from transformers import AutoModelForCausalLM, AutoTokenizer
from make_data import route_system
tok = AutoTokenizer.from_pretrained("work/fin-lora/merged")
model = AutoModelForCausalLM.from_pretrained("work/fin-lora/merged", dtype=torch.float16).cuda().eval()
system = route_system()
for name in ("route.jsonl", "route_hard.jsonl"):
  cases = [json.loads(l) for l in open(f"eval/{name}") if l.strip()]
  right, missed = 0, []
  for c in cases:
      msgs = [{"role": "system", "content": system}, {"role": "user", "content": c["text"]}]
      ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True, enable_thinking=False)["input_ids"].cuda()
      out = model.generate(ids, max_new_tokens=80, do_sample=False)
      reply = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
      try:
          got = json.loads(reply[reply.find("{"): reply.rfind("}") + 1])["mode"]
      except Exception:
          got = "?"
      if got == c["mode"]:
          right += 1
      else:
          missed.append((c["text"], c["mode"], got))
  print(f"{name}: {right}/{len(cases)}  (the rules get {51 if name == 'route.jsonl' else 70})")
  for m in missed:
      print("  missed:", m)

In [ ]:
# 6. Shrink to one GGUF file (about 0.5 GB)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp work/llama.cpp
# Not llama.cpp's requirements file: it pins an older transformers that
# can't read Qwen3.5's tokenizer. The converter brings its own gguf module.
!pip install -q sentencepiece
!python work/llama.cpp/convert_hf_to_gguf.py work/fin-lora/merged --outfile work/fin-f16.gguf --outtype f16
!cmake -S work/llama.cpp -B work/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build work/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!work/llama.cpp/build/bin/llama-quantize work/fin-f16.gguf work/fin-q4_k_m.gguf Q4_K_M
!ls -lh work/fin-q4_k_m.gguf

In [ ]:
# 7. Download it
from google.colab import files
files.download("work/fin-q4_k_m.gguf")

## On your Mac

Put the file somewhere (say `~/Downloads`), then:

```
brew install llama.cpp
llama-server -m ~/Downloads/fin-q4_k_m.gguf --port 8088 --alias fin --chat-template-kwargs '{"enable_thinking":false}' -c 4096 -np 1
```

The flags matter: Qwen3.5 thinks first by default and runs out of room before it answers, and without `-c 4096 -np 1` the server reserves 3.3 GB of memory instead of about 250 MB.

In `~/.squadcode/settings.json` set `"brain_model": "fin"`, and Squad uses it. `SquadCode v2 brain eval` scores it against the rules.